# Network Security Tool

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
from datetime import datetime, timedelta
import socket
import struct
import random
from collections import defaultdict
import ipaddress
import warnings
warnings.filterwarnings('ignore')

class NetworkSecurityScanner:
    def __init__(self):
        self.name = "Educational Network Security Assessment"
        self.vulnerabilities_db = self.load_vulnerability_database()
        self.generate_sample_network()

    def load_vulnerability_database(self):
        return {
            'CVE-2023-0001': {'severity': 'HIGH', 'type': 'Remote Code Execution', 'score': 9.8},
            'CVE-2023-0002': {'severity': 'MEDIUM', 'type': 'SQL Injection', 'score': 6.5},
            'CVE-2023-0003': {'severity': 'LOW', 'type': 'Information Disclosure', 'score': 3.1},
            'CVE-2023-0004': {'severity': 'CRITICAL', 'type': 'Buffer Overflow', 'score': 10.0},
            'CVE-2023-0005': {'severity': 'HIGH', 'type': 'Cross-Site Scripting', 'score': 8.2},
            'CVE-2023-0006': {'severity': 'MEDIUM', 'type': 'Directory Traversal', 'score': 5.8},
            'CVE-2023-0007': {'severity': 'HIGH', 'type': 'Authentication Bypass', 'score': 9.1},
            'CVE-2023-0008': {'severity': 'LOW', 'type': 'Weak Encryption', 'score': 4.0}
        }

    def generate_sample_network(self):
        np.random.seed(42)

        self.network_hosts = []
        for i in range(50):
            host = {
                'ip': f"192.168.1.{i+1}",
                'hostname': f"host-{i+1:02d}",
                'os': np.random.choice(['Windows 10', 'Ubuntu 20.04', 'CentOS 7', 'macOS'], p=[0.4, 0.3, 0.2, 0.1]),
                'services': self.generate_services(),
                'vulnerabilities': self.assign_vulnerabilities(),
                'last_updated': datetime.now() - timedelta(days=np.random.randint(1, 365)),
                'firewall_enabled': np.random.choice([True, False], p=[0.7, 0.3]),
                'antivirus_status': np.random.choice(['Active', 'Inactive', 'Outdated'], p=[0.6, 0.2, 0.2])
            }
            self.network_hosts.append(host)

    def generate_services(self):
        all_services = [
            {'port': 22, 'service': 'SSH', 'version': '7.4'},
            {'port': 80, 'service': 'HTTP', 'version': '2.4.41'},
            {'port': 443, 'service': 'HTTPS', 'version': '2.4.41'},
            {'port': 21, 'service': 'FTP', 'version': '2.3.4'},
            {'port': 25, 'service': 'SMTP', 'version': '2.11'},
            {'port': 53, 'service': 'DNS', 'version': '9.11'},
            {'port': 135, 'service': 'RPC', 'version': '1.0'},
            {'port': 445, 'service': 'SMB', 'version': '3.0'},
            {'port': 3389, 'service': 'RDP', 'version': '10.0'},
            {'port': 5432, 'service': 'PostgreSQL', 'version': '12.0'}
        ]

        num_services = np.random.randint(1, 6)
        return random.sample(all_services, num_services)

    def assign_vulnerabilities(self):
        vuln_count = np.random.poisson(2)
        if vuln_count == 0:
            return []

        vulns = random.sample(list(self.vulnerabilities_db.keys()), min(vuln_count, len(self.vulnerabilities_db)))
        return vulns

    def port_scan_simulation(self, target_ip):
        target_host = next((h for h in self.network_hosts if h['ip'] == target_ip), None)

        if not target_host:
            return {"error": "Host not found"}

        open_ports = []
        for service in target_host['services']:
            scan_result = {
                'port': service['port'],
                'service': service['service'],
                'version': service['version'],
                'state': 'open'
            }
            open_ports.append(scan_result)

        closed_ports = random.sample(range(1, 65535), 10)
        for port in closed_ports:
            if port not in [s['port'] for s in target_host['services']]:
                open_ports.append({
                    'port': port,
                    'service': 'unknown',
                    'version': 'unknown',
                    'state': 'closed'
                })

        return {
            'target': target_ip,
            'hostname': target_host['hostname'],
            'ports': sorted(open_ports, key=lambda x: x['port'])
        }

    def vulnerability_assessment(self, target_ip):
        target_host = next((h for h in self.network_hosts if h['ip'] == target_ip), None)

        if not target_host:
            return {"error": "Host not found"}

        assessment = {
            'target': target_ip,
            'hostname': target_host['hostname'],
            'os': target_host['os'],
            'last_updated': target_host['last_updated'].strftime('%Y-%m-%d'),
            'firewall_enabled': target_host['firewall_enabled'],
            'antivirus_status': target_host['antivirus_status'],
            'vulnerabilities': []
        }

        for vuln_id in target_host['vulnerabilities']:
            vuln_info = self.vulnerabilities_db[vuln_id].copy()
            vuln_info['cve_id'] = vuln_id
            assessment['vulnerabilities'].append(vuln_info)

        risk_score = self.calculate_risk_score(target_host)
        assessment['risk_score'] = risk_score
        assessment['risk_level'] = self.get_risk_level(risk_score)

        return assessment

    def calculate_risk_score(self, host):
        base_score = 0

        for vuln_id in host['vulnerabilities']:
            base_score += self.vulnerabilities_db[vuln_id]['score']

        if not host['firewall_enabled']:
            base_score += 2

        if host['antivirus_status'] == 'Inactive':
            base_score += 3
        elif host['antivirus_status'] == 'Outdated':
            base_score += 1.5

        days_since_update = (datetime.now() - host['last_updated']).days
        if days_since_update > 90:
            base_score += 2
        elif days_since_update > 30:
            base_score += 1

        return min(base_score, 10.0)

    def get_risk_level(self, score):
        if score >= 9.0:
            return 'CRITICAL'
        elif score >= 7.0:
            return 'HIGH'
        elif score >= 4.0:
            return 'MEDIUM'
        else:
            return 'LOW'

    def network_discovery(self):
        discovered_hosts = []

        for host in self.network_hosts[:20]:
            discovery_info = {
                'ip': host['ip'],
                'hostname': host['hostname'],
                'os': host['os'],
                'response_time': np.random.exponential(50),
                'mac_address': self.generate_mac_address(),
                'vendor': np.random.choice(['Dell', 'HP', 'Cisco', 'Apple', 'Lenovo'])
            }
            discovered_hosts.append(discovery_info)

        return discovered_hosts

    def generate_mac_address(self):
        return ':'.join(['%02x' % random.randint(0, 255) for _ in range(6)])

    def security_compliance_check(self):
        compliance_results = {
            'total_hosts': len(self.network_hosts),
            'firewall_compliance': 0,
            'antivirus_compliance': 0,
            'update_compliance': 0,
            'vulnerability_count': 0,
            'critical_vulnerabilities': 0
        }

        for host in self.network_hosts:
            if host['firewall_enabled']:
                compliance_results['firewall_compliance'] += 1

            if host['antivirus_status'] == 'Active':
                compliance_results['antivirus_compliance'] += 1

            days_since_update = (datetime.now() - host['last_updated']).days
            if days_since_update <= 30:
                compliance_results['update_compliance'] += 1

            compliance_results['vulnerability_count'] += len(host['vulnerabilities'])

            for vuln_id in host['vulnerabilities']:
                if self.vulnerabilities_db[vuln_id]['severity'] == 'CRITICAL':
                    compliance_results['critical_vulnerabilities'] += 1

        total_hosts = compliance_results['total_hosts']
        compliance_results['firewall_percentage'] = (compliance_results['firewall_compliance'] / total_hosts) * 100
        compliance_results['antivirus_percentage'] = (compliance_results['antivirus_compliance'] / total_hosts) * 100
        compliance_results['update_percentage'] = (compliance_results['update_compliance'] / total_hosts) * 100

        return compliance_results

    def generate_security_report(self):
        compliance = self.security_compliance_check()

        risk_distribution = defaultdict(int)
        for host in self.network_hosts:
            risk_score = self.calculate_risk_score(host)
            risk_level = self.get_risk_level(risk_score)
            risk_distribution[risk_level] += 1

        vuln_severity_count = defaultdict(int)
        for host in self.network_hosts:
            for vuln_id in host['vulnerabilities']:
                severity = self.vulnerabilities_db[vuln_id]['severity']
                vuln_severity_count[severity] += 1

        fig, axes = plt.subplots(2, 2, figsize=(15, 12))

        compliance_metrics = ['Firewall', 'Antivirus', 'Updates']
        compliance_values = [compliance['firewall_percentage'], compliance['antivirus_percentage'], compliance['update_percentage']]
        colors = ['green' if x >= 80 else 'orange' if x >= 60 else 'red' for x in compliance_values]

        axes[0,0].bar(compliance_metrics, compliance_values, color=colors)
        axes[0,0].set_title('Security Compliance Percentage')
        axes[0,0].set_ylabel('Compliance %')
        axes[0,0].axhline(y=80, color='green', linestyle='--', alpha=0.5, label='Target: 80%')
        axes[0,0].legend()

        risk_levels = list(risk_distribution.keys())
        risk_counts = list(risk_distribution.values())
        risk_colors = {'CRITICAL': 'red', 'HIGH': 'orange', 'MEDIUM': 'yellow', 'LOW': 'green'}
        colors = [risk_colors.get(level, 'gray') for level in risk_levels]

        axes[0,1].pie(risk_counts, labels=risk_levels, colors=colors, autopct='%1.1f%%')
        axes[0,1].set_title('Risk Level Distribution')

        severities = list(vuln_severity_count.keys())
        severity_counts = list(vuln_severity_count.values())
        severity_colors = {'CRITICAL': 'darkred', 'HIGH': 'red', 'MEDIUM': 'orange', 'LOW': 'yellow'}
        colors = [severity_colors.get(sev, 'gray') for sev in severities]

        axes[1,0].bar(severities, severity_counts, color=colors)
        axes[1,0].set_title('Vulnerability Severity Distribution')
        axes[1,0].set_ylabel('Count')

        top_vulnerable_hosts = sorted(self.network_hosts, key=lambda x: len(x['vulnerabilities']), reverse=True)[:10]
        host_ips = [h['ip'] for h in top_vulnerable_hosts]
        vuln_counts = [len(h['vulnerabilities']) for h in top_vulnerable_hosts]

        axes[1,1].barh(host_ips, vuln_counts, color='coral')
        axes[1,1].set_title('Top 10 Most Vulnerable Hosts')
        axes[1,1].set_xlabel('Vulnerability Count')

        plt.tight_layout()
        plt.show()

        return {
            'compliance': compliance,
            'risk_distribution': dict(risk_distribution),
            'vulnerability_summary': dict(vuln_severity_count)
        }

    def security_recommendations(self, assessment_results):
        recommendations = []

        if assessment_results['compliance']['firewall_percentage'] < 80:
            recommendations.append("🔥 Enable firewalls on all network hosts")

        if assessment_results['compliance']['antivirus_percentage'] < 80:
            recommendations.append("🛡️ Deploy and maintain active antivirus on all systems")

        if assessment_results['compliance']['update_percentage'] < 70:
            recommendations.append("🔄 Implement automated patch management system")

        if assessment_results['compliance']['critical_vulnerabilities'] > 0:
            recommendations.append("⚠️ URGENT: Address all critical vulnerabilities immediately")

        if assessment_results['risk_distribution'].get('HIGH', 0) > 10:
            recommendations.append("📊 Conduct risk assessment for high-risk systems")

        recommendations.extend([
            "🔐 Implement network segmentation",
            "👥 Conduct security awareness training",
            "📝 Establish incident response procedures",
            "🔍 Deploy continuous monitoring solutions",
            "🔒 Implement multi-factor authentication"
        ])

        return recommendations

    def run_comprehensive_scan(self):
        print("🔍 NETWORK SECURITY ASSESSMENT TOOL")
        print("=" * 50)

        print("\n1. NETWORK DISCOVERY:")
        discovered = self.network_discovery()
        print(f"Discovered {len(discovered)} active hosts")
        for host in discovered[:5]:
            print(f"  {host['ip']} ({host['hostname']}) - {host['os']}")

        print("\n2. PORT SCAN EXAMPLE:")
        sample_ip = self.network_hosts[0]['ip']
        scan_result = self.port_scan_simulation(sample_ip)
        print(f"Scanning {sample_ip}:")
        open_services = [p for p in scan_result['ports'] if p['state'] == 'open']
        for service in open_services[:5]:
            print(f"  Port {service['port']}: {service['service']} v{service['version']}")

        print("\n3. VULNERABILITY ASSESSMENT:")
        vuln_result = self.vulnerability_assessment(sample_ip)
        print(f"Host: {vuln_result['hostname']} ({vuln_result['target']})")
        print(f"Risk Level: {vuln_result['risk_level']} (Score: {vuln_result['risk_score']:.1f})")
        print(f"Vulnerabilities found: {len(vuln_result['vulnerabilities'])}")

        print("\n4. SECURITY COMPLIANCE REPORT:")
        report_data = self.generate_security_report()

        print("\n5. SECURITY RECOMMENDATIONS:")
        recommendations = self.security_recommendations(report_data)
        for i, rec in enumerate(recommendations[:8], 1):
            print(f"  {i}. {rec}")

        return report_data

scanner = NetworkSecurityScanner()
results = scanner.run_comprehensive_scan()

print(f"\n🎯 INTERACTIVE FEATURES:")
print("Try these commands:")
print("• scanner.port_scan_simulation('192.168.1.5')")
print("• scanner.vulnerability_assessment('192.168.1.10')")
print("• scanner.network_discovery()")
print("• scanner.security_compliance_check()")